In [1]:
import pandas as pd

df = pd.read_csv('./yolo_update_data.csv')

In [2]:
df.columns

Index(['Unnamed: 0', 'keyword', 'source_file', 'asin', 'item_name', 'brand',
       'image_count', 'main_image_url', 'has_aplus', 'has_brand_story',
       'bsr_best', 'bsr_paths', 'product_url', 'image_list', 'image_path',
       'edge_density', 'bg_white_pct', 'bg_neutral_pct', 'n_clusters_sig',
       'color_entropy', 'largest_cluster_pct', 'edge_density_z',
       'n_clusters_sig_z', 'color_entropy_z', 'bg_white_pct_z',
       'bg_neutral_pct_z', 'largest_cluster_pct_z', 'clutter_score',
       'n_objects', 'category_diversity', 'has_person', 'sum_box_ratio',
       'product_box_ratio', 'center_offset_main', 'mean_iou_overlap',
       'fg_ratio_grabcut', 'center_offset_grabcut', 'product_coverage',
       'center_offset', 'white_bg_ok'],
      dtype='object')

In [3]:
df['image_path']

0        images_amz/3347156682c8f0ca.jpg
1        images_amz/e7c4b57eac5ca716.jpg
2        images_amz/e7c4b57eac5ca716.jpg
3        images_amz/a114c60191cfaf62.jpg
4        images_amz/a114c60191cfaf62.jpg
                      ...               
19044    images_amz/116571ed183ac133.jpg
19045    images_amz/e3e25738baf8b982.jpg
19046    images_amz/6f8328802945bc64.jpg
19047    images_amz/5bae15f9e5ecc844.jpg
19048    images_amz/9077ea2f246e7ce9.jpg
Name: image_path, Length: 19049, dtype: object

In [4]:
# pip install pyarrow tqdm
import os, gc
import cv2
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# limit native threads (prevents oversubscription crashes)
try:
    cv2.setNumThreads(1)
except Exception:
    pass

# --- placeholder for your image folder ---
IMG_DIR = "./images_amz"  # change if needed

# Where to save batch outputs (so you can resume)
OUT_DIR = "tech_quality_batches"
os.makedirs(OUT_DIR, exist_ok=True)

def load_cv2_safe(path):
    try:
        data = np.fromfile(path, dtype=np.uint8)
        if data.size == 0:
            return None
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        return img
    except Exception:
        return None

def downscale_max_side(img, max_side=1024):
    h, w = img.shape[:2]
    s = max(h, w)
    if s <= max_side:
        return img
    scale = max_side / float(s)
    nh, nw = int(h * scale), int(w * scale)
    return cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)

def tech_quality_for_path(p):
    img = load_cv2_safe(p)
    if img is None:
        return dict(image_path=p, sharpness=np.nan, exposure=np.nan,
                    contrast=np.nan, saturation=np.nan, noise=np.nan)

    # downscale early to reduce RAM/compute
    img = downscale_max_side(img, max_side=1024)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Sharpness: variance of Laplacian
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    # Exposure & Contrast
    exposure = float(gray.mean())
    contrast = float(gray.std())

    # Saturation
    saturation = float(hsv[..., 1].mean())

    # Noise proxy: diff to lightly blurred image
    blur = cv2.GaussianBlur(img, (3,3), 0)
    noise = float(np.mean(np.abs(img.astype(np.float32) - blur.astype(np.float32))))

    return dict(image_path=p, sharpness=sharpness, exposure=exposure,
                contrast=contrast, saturation=saturation, noise=noise)

# --- main batched runner (resumable) ---
paths_all = df["image_path"].dropna().tolist()

# resume support: skip paths already processed in existing batch files
processed_paths = set()
for f in os.listdir(OUT_DIR):
    if f.endswith(".parquet"):
        try:
            part = pd.read_parquet(os.path.join(OUT_DIR, f), columns=["image_path"])
            processed_paths.update(part["image_path"].tolist())
        except Exception:
            pass

paths = [p for p in paths_all if p not in processed_paths]

BATCH_SIZE = 500         # tune based on RAM
MAX_WORKERS =   4         # small thread pool to overlap I/O safely

for start in tqdm(range(0, len(paths), BATCH_SIZE), desc="Batches"):
    chunk = paths[start:start+BATCH_SIZE]
    rows = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(tech_quality_for_path, p): p for p in chunk}
        for fut in tqdm(as_completed(futures), total=len(chunk), leave=False, desc="Images"):
            rows.append(fut.result())

    df_tech_batch = pd.DataFrame(rows)
    out_path = os.path.join(OUT_DIR, f"tech_batch_{start:07d}.parquet")
    df_tech_batch.to_parquet(out_path, index=False)

    # free memory
    del rows, df_tech_batch
    gc.collect()

# combine all batches
parts = []
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith(".parquet"):
        parts.append(pd.read_parquet(os.path.join(OUT_DIR, f)))
df_tech = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=[
    "image_path","sharpness","exposure","contrast","saturation","noise"
])

# merge into df (no reprocessing needed if kernel dies; just rerun combine+merge)
df = df.merge(df_tech, on="image_path", how="left")

# optional composite z-scores (vectorized; cheap)
for c in ["sharpness","contrast","saturation","noise","exposure"]:
    m, s = df[c].mean(), df[c].std(ddof=0) + 1e-9
    df[c+"_z"] = (df[c] - m) / s

df["tech_quality_score"] = (
    + 0.5  * df["sharpness_z"]
    + 0.25 * df["contrast_z"]
    + 0.15 * df["saturation_z"]
    - 0.10 * df["noise_z"]
    - 0.05 * df["exposure_z"].abs()
)


/opt/anaconda3/envs/amazon-forecast/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 36/36 [03:20<00:00,  5.57s/it]


In [9]:
df

,Unnamed: 0,keyword,source_file,asin,item_name,brand,image_count,main_image_url,has_aplus,has_brand_story,...,exposure,contrast,saturation,noise,sharpness_z,contrast_z,saturation_z,noise_z,exposure_z,tech_quality_score
0,0,audio headphones catalog full 1757641194,audio_headphones_catalog_full_1757641194.csv,B0BQPNMXQV,JBL Vibe Beam - True Wireless JBL Deep Bass So...,JBL,18,https://m.media-amazon.com/images/I/31S4tOQj4S...,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,audio headphones catalog full 1757641194,audio_headphones_catalog_full_1757641194.csv,B0CTBCDD6D,JBL Tune 720BT - Wireless Over-Ear Headphones ...,JBL,21,https://m.media-amazon.com/images/I/61EL2AKKcB...,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,audio headphones catalog full 1757641194,audio_headphones_catalog_full_1757641194.csv,B0CTBCDD6D,JBL Tune 720BT - Wireless Over-Ear Headphones ...,JBL,21,https://m.media-amazon.com/images/I/61EL2AKKcB...,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,audio headphones catalog full 1757641194,audio_headphones_catalog_full_1757641194.csv,B08WM3LMJF,JBL Tune 510BT - Bluetooth headphones with up ...,JBL,24,https://m.media-amazon.com/images/I/61kFL7ywsZ...,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,audio headphones catalog full 1757641194,audio_headphones_catalog_full_1757641194.csv,B08WM3LMJF,JBL Tune 510BT - Bluetooth headphones with up ...,JBL,24,https://m.media-amazon.com/images/I/61kFL7ywsZ...,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50136,19044,tv catalog full 1757643738,tv_catalog_full_1757643738.csv,B0F2QVR6TH,SAMSUNG 48-Inch Class OLED S90F 4K Smart TV (2...,Samsung,27,https://m.media-amazon.com/images/I/61g9-+wOuK...,False,False,...,152.822031,98.216529,72.955547,1.706521,-0.481702,1.145833,0.947057,-0.257495,-0.914649,0.167683
50137,19045,tv catalog full 1757643738,tv_catalog_full_1757643738.csv,B0DF85L4S8,Sony K77XR80 77 Inch IMAX Enhanced Bravia OLED...,Sony,27,https://m.media-amazon.com/images/I/71wA-vGqqo...,False,False,...,164.529679,97.503845,89.202922,1.824577,0.061339,1.109403,1.372283,-0.170705,-0.584819,0.501692
50138,19046,tv catalog full 1757643738,tv_catalog_full_1757643738.csv,B0FMC49YWD,Panasonic TV-77Z95BP Z95BP Series 77 inch LED ...,Panasonic,27,https://m.media-amazon.com/images/I/71cUpjnhqM...,False,False,...,114.625145,101.667018,83.286773,4.244010,1.450257,1.322208,1.217445,1.607965,-1.990736,0.977964
50139,19047,tv catalog full 1757643738,tv_catalog_full_1757643738.csv,B0DDYBNHV5,Sony 42 Inch 4K Ultra HD TV A90K Series: BRAVI...,Sony,3,https://m.media-amazon.com/images/I/71g4cqhMQ8...,False,False,...,231.734986,66.147484,21.566112,0.853131,-0.461556,-0.493406,-0.397905,-0.884873,1.308496,-0.390753
